# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-28: ECG-Fitness Exploration

### What this notebook does
Characterises ECG-Fitness before anything is preprocessed: the on-disk layout, the two
clocks the dataset carries, the reference signal, the real frame timing, and the face
geometry. Almost all of it runs from the small csv and bounding-box files, without
decoding a frame, so the corpus-wide picture costs seconds rather than the hours a
1.1 TB raw-video pass would. Only sections 6 and 7 open a video, on a named subset.

### Dataset
Spetlik, Franc, Cech, Matas, *Visual Heart Rate Estimation with Convolutional Neural
Network* (BMVC 2018). 17 subjects on fitness machines, each recorded in six sessions by
two Logitech C920 cameras at once, 60 s per session. Access is by signed request to the
Center for Machine Perception, CTU Prague, and **carries no redistribution right**.

Per session folder `XX/YY/`:
- `c920-1.avi`, `c920-2.avi` - uncompressed YUV 4:2:0, 1920x1080, ~5.6 GB each. OpenCV
  decodes them to BGR on its own; the `COLOR_YUV2RGB_I420` flag crashes.
- `c920.csv` - one row per video frame: a wall-clock timestamp and a row index into the
  reference file.
- `viatom-raw.csv` - the Viatom Checkme recording: timestamp, ECG, device HR, and PPG /
  SpO2 / PI columns.
- `bbox/XX/YY/c920-*.face` - the authors' per-frame face box, four numbers per frame.

### Role in Phase 3
ECG-Fitness is **not** training data. Its reference is an ECG with no usable
photoplethysmographic waveform, and PhysNet v2 is trained against a BVP waveform, so
there is nothing here for the loss to match. Its value is the opposite one: this is the
only corpus in reach where high heart rate and violent motion arrive together, which
makes it a **refusal benchmark** - does the app decline when it cannot see a pulse, or
does it report a confident number that happens to be wrong? The failure that matters is
a window the app calls `ok` and misses by more than 10 bpm.

That also bounds what may be claimed from it. High HR and motion are confounded here, so
nothing in this corpus isolates a high-HR bias; MCD-rPPG's post-exercise recordings, with
the subject still, remain the cleaner instrument for that question.

### What this notebook has to settle
1. **Is the reference usable, and on which recordings?** Phase 2 reported an ECG SQI of
   0.93 against a POS SQI of 0.22-0.28, i.e. the video side failed, not the reference.
   That needs re-deriving on the app's own window grid before any gate is pre-registered.
2. **Which camera?** Phase 2 used `c920-1` as the frontal one, without a recorded
   comparison. The authors' bounding boxes answer it at no cost.
3. **Is the halogen exclusion right?** NB_P2_01 dropped activities 03 and 04 on the
   grounds that 50 Hz mains flicker aliases to 3.33 Hz, inside the cardiac band. Section 6
   shows that arithmetic does not hold and measures the thing itself.
4. **Does our own detector see these faces?** Phase 2 recorded that subjects 01, 03, 06,
   11 and 14 passed nothing. If MediaPipe finds no face, a refusal benchmark on those
   recordings measures the detector, not the estimator, and they have to be reported
   separately.
5. **Does a store need building at all**, and how large is it.

No face or frame is rendered anywhere in this notebook.

## Imports & Setup

In [1]:
import re
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks

# Walk up from the notebook until the repo root is found, identified by the MediaPipe
# asset (the same test NB_P3_25 and NB_P3_27 use; cwd is the nested notebook folder).
MP_REL = Path('app/live_hr_demo/models/mediapipe/face_landmarker.task')
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / MP_REL).exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'no {MP_REL} in any parent of {Path.cwd()}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# The app's contract is imported, not restated. Every gate this notebook reports against
# is the one a benchmark would actually apply, and it moves if config.py moves.
from app.live_vitals.config import (CLIP_LEN, CROP_PADDING, MAX_BOX_SIDE_LOST,
                                    MIN_FPS_ACCEPT, MIN_FPS_WARN, N_DETECT_FRAMES,
                                    TARGET_FPS, TARGET_SIZE, TRAIN_ASPECT_RANGE,
                                    WINDOW_STRIDE)

ECGF_ROOT = Path(r'F:\ECG_Fitness_dataset\ecg-fitness_raw-v1.0')
BBOX_ROOT = ECGF_ROOT / 'bbox'
CAMERAS = ('c920-1', 'c920-2')

FRAME_W, FRAME_H = 1920, 1080  # confirmed against the published sample frames
ECG_SCALE = 65536.0            # the ECG column is fixed point; section 2 verifies it
VIATOM_FS_NOMINAL = 125.0      # measured per recording in section 3, never assumed
HR_MIN_BPM, HR_MAX_BPM = 35.0, 220.0  # physiological limits on a beat interval
QRS_LOW_HZ, QRS_HIGH_HZ = 5.0, 20.0   # QRS energy band for the R-peak detector
HR_AGREE_BPM = 3.0             # the agreement bar NB_P3_27 used between references
MAINS_HZ = 50.0                # Czech mains; a lamp's brightness peaks twice per cycle

# Activity names are deliberately absent. NB_P2_01 labelled 01 'rowing' and 02
# 'speaking', then measured 83.8 bpm with 9/17 recordings passing its gate for 01 and
# 115.8 bpm with 1/17 for 02 - the wrong way round for those two activities. Until the
# dataset manual is read, activities are identified by id and characterised from the
# data (sections 4 and 5), never named.
ACTIVITY_IDS = ('01', '02', '03', '04', '05', '06')

print('ECG-Fitness root:', ECGF_ROOT, '| exists:', ECGF_ROOT.exists())
print('author bounding boxes:', BBOX_ROOT.exists())
print(f'app contract: clip {CLIP_LEN} frames at {TARGET_FPS:g} fps = {CLIP_LEN / TARGET_FPS:.2f} s, '
      f'stride {WINDOW_STRIDE} | crop padding {CROP_PADDING} | '
      f'rate gate warn {MIN_FPS_WARN:g} / accept {MIN_FPS_ACCEPT:g} fps')

ECG-Fitness root: F:\ECG_Fitness_dataset\ecg-fitness_raw-v1.0 | exists: True
author bounding boxes: True
app contract: clip 160 frames at 30 fps = 5.33 s, stride 80 | crop padding 0.6 | rate gate warn 20 / accept 27 fps


## 1. Inventory
Counts what is actually on the disk and checks it against what the dataset documents as
missing, before any of it is trusted. Three things are worth separating: a session with
no video at all, a session whose video exists but whose author bounding box does not, and
a session missing one of the two small reference files. Only the first removes a
recording from the benchmark; the second removes the cheap face audit in section 5 for
that recording; the third would remove the reference.

In [2]:
TWO_DIGIT = re.compile(r'^\d{2}$')


def scan_sessions(root=ECGF_ROOT):
    """Returns one row per subject/activity folder with file presence and video size."""
    rows = []
    for subj in sorted(p for p in root.iterdir() if p.is_dir() and TWO_DIGIT.match(p.name)):
        for act in sorted(p for p in subj.iterdir() if p.is_dir() and TWO_DIGIT.match(p.name)):
            row = {'subject': subj.name, 'activity': act.name}
            for cam in CAMERAS:
                video = act / f'{cam}.avi'
                row[f'{cam}_gb'] = video.stat().st_size / 1e9 if video.exists() else np.nan
                row[f'{cam}_bbox'] = (BBOX_ROOT / subj.name / act.name / f'{cam}.face').exists()
            row['sync'] = (act / 'c920.csv').exists()
            row['viatom'] = (act / 'viatom-raw.csv').exists()
            rows.append(row)
    return pd.DataFrame(rows)


sessions = scan_sessions()
size_cols = [f'{c}_gb' for c in CAMERAS]
expected = {(s, a) for s in sessions.subject.unique() for a in ACTIVITY_IDS}
missing_session = sorted(expected - set(zip(sessions.subject, sessions.activity)))
missing_bbox = [(r.subject, r.activity, c) for _, r in sessions.iterrows() for c in CAMERAS
                if np.isfinite(r[f'{c}_gb']) and not r[f'{c}_bbox']]
missing_ref = sessions.loc[~(sessions.sync & sessions.viatom), ['subject', 'activity']]

print(f'subjects {sessions.subject.nunique()} | sessions {len(sessions)} '
      f'| videos {int(sessions[size_cols].notna().to_numpy().sum())} '
      f'| {sessions[size_cols].sum().sum():.0f} GB of raw video')
print('sessions with no video      :', missing_session or 'none')
print('videos with no author bbox  :', missing_bbox or 'none')
print('sessions missing a reference:', missing_ref.to_dict('records') or 'none')
print()
print('video size per camera (GB):')
print(sessions[size_cols].describe().loc[['min', '50%', 'max']].round(2).to_string())
print()
print('sessions per subject:', sessions.groupby('subject').size().to_dict())

subjects 17 | sessions 101 | videos 202 | 1127 GB of raw video
sessions with no video      : [('02', '04')]
videos with no author bbox  : [('07', '02', 'c920-1'), ('07', '02', 'c920-2')]
sessions missing a reference: none

video size per camera (GB):
     c920-1_gb  c920-2_gb
min        3.8        3.8
50%        5.6        5.6
max        5.6        5.6

sessions per subject: {'00': 6, '01': 6, '02': 5, '03': 6, '04': 6, '05': 6, '06': 6, '07': 6, '08': 6, '09': 6, '10': 6, '11': 6, '12': 6, '13': 6, '14': 6, '15': 6, '16': 6}


## 2. Two clocks, and what the sync file actually is
This is the step that decides whether anything downstream is aligned, so it is done on
one recording in full before it is run over the corpus.

ECG-Fitness carries **two independent clocks**: the camera's and the Viatom's. The
reference recording starts before the video and runs longer than it, so the naive
assumption that works for UBFC-rPPG - reference and video span the same window, align by
proportional resampling - is wrong here and would shift every label by however long the
operator waited before starting the camera.

`c920.csv` is what bridges them, and it has **two columns**, not one: a wall-clock
timestamp per frame, and the row index into `viatom-raw.csv` for that frame. NB_P2_01
read it with `pd.read_csv(...).values.flatten()`, which interleaves the two columns into
a single array twice the length of the video; after its `[1:]` the values alternate
between a reference index and a 13-digit epoch timestamp, and the array it fed to the
aligner was twice as long as the video. Its frame-level `hr_continuous` was therefore
built against the wrong clock on every second frame. The per-recording `hr_mean` came
from R-peaks and is unaffected; whether any Phase-2 headline number depended on the
frame-aligned trace has not been checked here. Either way, nothing frame-aligned from
Phase 2 should be reused.

The timestamp column is also a gift. Every other corpus here needs the frame rate
inferred, and believing a container's declared fps has gone wrong before: three MCD
recordings declare 24.00 at a true ~29.7, which scales heart rate by 0.8. Here the true
rate can be read straight off the capture clock.

The cell below also checks the ECG column's scaling - every value is a multiple of
2^16 - and which of the Viatom's seven columns carry a signal at all.

In [3]:
def read_viatom(path):
    """Reads viatom-raw.csv, dropping any row that did not parse completely.

    One recording (07/02) ends mid-line, and pandas turns that fragment into a row
    carrying a timestamp and nothing else. Left in, a single row makes the file's
    apparent span 1.4e10 s, its sampling rate 6e-07 Hz and its dead PPG column look
    alive. The count of dropped rows is kept on the frame so the audit can report it
    rather than silently repair it.
    """
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    complete = df.notna().all(axis=1)
    out = df.loc[complete].astype('int64').reset_index(drop=True)
    out.attrs['dropped_rows'] = int((~complete).sum())
    return out


def read_sync(path):
    """Reads c920.csv as (frame wall-clock ms, Viatom row index), one row per frame."""
    a = np.loadtxt(path, delimiter=',', dtype=np.int64)
    return a[:, 0], a[:, 1]


def device_hr(df):
    """Decodes the Viatom 'ECG HR' column, which is a signed byte and wraps.

    Any rate above 127 is stored negative: 135 bpm reads as -121. It goes negative in
    45 of the 100 usable recordings, and those are the fast ones - exactly the part of
    this corpus the benchmark exists for - so reading the column as written would lose
    the independent cross-check precisely where it matters most. After unwrapping,
    values outside the physiological band (0 while the device is still computing, 255
    from its -1 'no reading' sentinel) become NaN.
    """
    hr = df['ECG HR'].to_numpy() % 256
    return np.where((hr >= HR_MIN_BPM) & (hr <= HR_MAX_BPM), hr.astype(float), np.nan)


demo_subject, demo_activity = '00', '01'
demo = ECGF_ROOT / demo_subject / demo_activity
v = read_viatom(demo / 'viatom-raw.csv')
t_frame, idx = read_sync(demo / 'c920.csv')
t_ref = v['milliseconds'].to_numpy()
dev = v['ECG HR'].to_numpy()

print(f'--- {demo_subject}/{demo_activity} ---')
print('viatom-raw.csv columns:', list(v.columns), '| rows:', len(v))
print(v.head(3).to_string(index=False))
print()
print('c920.csv: shape', (len(t_frame), 2), '| first rows:',
      list(zip(t_frame[:3].tolist(), idx[:3].tolist())))
print()

ref_fs = (len(v) - 1) / ((t_ref[-1] - t_ref[0]) / 1000)
vid_fps = (len(t_frame) - 1) / ((t_frame[-1] - t_frame[0]) / 1000)
offset_s = (t_frame[0] - t_ref[0]) / 1000
print(f'reference: {len(v)} rows over {(t_ref[-1] - t_ref[0]) / 1000:6.2f} s -> {ref_fs:7.3f} Hz '
      f'({len(np.unique(t_ref))} distinct timestamps, written in blocks of '
      f'{len(v) / len(np.unique(t_ref)):.0f})')
print(f'video    : {len(t_frame)} frames over {(t_frame[-1] - t_frame[0]) / 1000:6.2f} s -> '
      f'{vid_fps:7.3f} fps')
print(f'the video starts {offset_s:.2f} s into the reference. Sync index 0 is {idx[0]}, '
      f'which at {ref_fs:.2f} Hz is {idx[0] / ref_fs:.2f} s - the two clocks agree to '
      f'{abs(offset_s - idx[0] / ref_fs) * 1000:.0f} ms.')
print()

ecg = v['ECG'].to_numpy()
print(f'ECG column: every value a multiple of {ECG_SCALE:.0f}: {bool(np.all(ecg % ECG_SCALE == 0))} '
      f'| raw {ecg.min()}..{ecg.max()} -> scaled {ecg.min() / ECG_SCALE:.0f}..'
      f'{ecg.max() / ECG_SCALE:.0f} over {len(np.unique(ecg / ECG_SCALE))} distinct levels')
for col in ('PPG', 'PPG HR', 'SpO2', 'PI'):
    a = v[col].to_numpy()
    uniq = np.unique(a)
    print(f'  {col:7s}: {len(uniq)} distinct value(s)' +
          (f', constant at {uniq[0]}' if len(uniq) == 1 else f', range {a.min()}..{a.max()}'))
print(f'  ECG HR : {len(np.unique(dev))} distinct values, range {dev.min()}..{dev.max()} '
      f'as written -> {np.nanmin(device_hr(v)):.0f}..{np.nanmax(device_hr(v)):.0f} bpm unwrapped')

--- 00/01 ---
viatom-raw.csv columns: ['milliseconds', 'ECG', 'ECG HR', 'PPG', 'PPG HR', 'SpO2', 'PI'] | rows: 8720
 milliseconds    ECG  ECG HR  PPG  PPG HR  SpO2  PI
1523629844327 589824      85  -56      -1    -1   0
1523629844327 393216      85  -56      -1    -1   0
1523629844327 327680      85  -56      -1    -1   0

c920.csv: shape (1800, 2) | first rows: [(1523629854001, 1210), (1523629854043, 1215), (1523629854062, 1220)]

reference: 8720 rows over  69.72 s -> 125.059 Hz (1744 distinct timestamps, written in blocks of 5)
video    : 1800 frames over  59.95 s ->  30.009 fps
the video starts 9.67 s into the reference. Sync index 0 is 1210, which at 125.06 Hz is 9.68 s - the two clocks agree to 1 ms.

ECG column: every value a multiple of 65536: True | raw -8847360..3604480 -> scaled -135..55 over 179 distinct levels
  PPG    : 1 distinct value(s), constant at -56
  PPG HR : 1 distinct value(s), constant at -1
  SpO2   : 1 distinct value(s), constant at -1
  PI     : 1 distinct va

## 3. Frame timing and the reference, across the corpus
The same reads over every session. This touches only the csv files - about 45 MB in
total - so it is a minute of disk, not a pass over the video.

Four things are being established.

**The true frame rate**, from the capture clock rather than the container. The app
refuses a capture below `MIN_FPS_WARN` and flags one below `MIN_FPS_ACCEPT`, and a wrong
time base is the one error spectral confidence cannot detect: a 10 fps capture once read
a confident ~7 bpm low.

**The jitter inside a nominally 30 fps capture.** The Phase-3 schema stores frames as a
uniform sequence with a single `fps` attribute, so any irregularity in arrival times
becomes phase noise in the stored waveform. `grid_rms_ms` measures the deviation from
the uniform grid a store would impose. It is reported rather than assumed away, because
a tenth of a frame of jitter is harmless for a spectral HR readout and fatal for beat
timing.

**The offset between the two clocks**, per recording, and whether `c920.csv` stays
consistent with both.

**Whether any recording has a live PPG.** This is the question that decides whether
ECG-Fitness could ever be training data. If the Viatom's PPG columns are dead in every
session, the refusal-benchmark role is the only one available.

Two defects turned up here on the first run, and both are handled rather than hidden.

**07/02 is truncated.** Its last line stops mid-record, which pandas reads as a row with
a timestamp and nothing else. That one row gave the file a 1.4e10 s span, a sampling
rate of 6e-07 Hz, an ECG column that fails the fixed-point test, and a PPG column that
appeared to carry two distinct values. The near-zero rate is what crashed the filter
design in section 4. It is also the one session with no author bounding box, so the
dataset's own authors excluded it: 100 usable sessions, not 101. The single apparently
live PPG in the corpus was this artefact, which settles the training question - there is
no photoplethysmographic waveform anywhere in ECG-Fitness.

**The device HR column wraps.** `ECG HR` is a signed byte, so anything above 127 is
written negative. It goes negative in 45 of the 100 usable recordings, and those are the
fast ones. Read as written, per-session medians came out as low as -121 bpm; unwrapped,
the same column reads 55-153 bpm and agrees with the beat-detected rate. See `device_hr`
in section 2.

A session is excluded here only for a defect that makes its reference unreadable, and
the reason is printed. Everything else is reported and left in.

In [4]:
def grid_deviation_ms(t_ms):
    """Deviation of real frame times from the uniform grid a Phase-3 store would impose.

    Returns (rms, max absolute) in milliseconds. The store keeps frames as a uniform
    sequence with one fps attribute, so this is the timing error that preprocessing
    silently introduces - worth knowing before a beat-level claim is made from it.
    """
    grid = np.linspace(t_ms[0], t_ms[-1], len(t_ms))
    d = t_ms - grid
    return float(np.sqrt(np.mean(d ** 2))), float(np.max(np.abs(d)))


REF_FS_RANGE = (100.0, 150.0)  # a Viatom recording outside this is not a 125 Hz record


def audit_session(subject, activity):
    """Timing and reference integrity for one session, from the csv files alone."""
    folder = ECGF_ROOT / subject / activity
    t_frame, idx = read_sync(folder / 'c920.csv')
    v = read_viatom(folder / 'viatom-raw.csv')
    t_ref = v['milliseconds'].to_numpy()
    ecg = v['ECG'].to_numpy()
    dev = device_hr(v)
    dt = np.diff(t_frame)
    span_s = (t_frame[-1] - t_frame[0]) / 1000
    ref_span_s = (t_ref[-1] - t_ref[0]) / 1000
    ref_fs = (len(v) - 1) / ref_span_s if ref_span_s > 0 else np.nan
    offset_s = (t_frame[0] - t_ref[0]) / 1000
    rms_ms, max_ms = grid_deviation_ms(t_frame.astype(float))
    row = {
        'subject': subject, 'activity': activity,
        'n_frames': len(t_frame), 'span_s': span_s, 'fps': (len(t_frame) - 1) / span_s,
        'dt_med_ms': float(np.median(dt)), 'dt_max_ms': float(dt.max()),
        'dt_gt_50ms': int((dt > 50).sum()),
        'grid_rms_ms': rms_ms, 'grid_max_ms': max_ms,
        'dropped_rows': int(v.attrs.get('dropped_rows', 0)),
        'ref_rows': len(v), 'ref_span_s': ref_span_s, 'ref_fs': ref_fs,
        'offset_s': offset_s,
        'clock_gap_ms': abs(offset_s - idx[0] / ref_fs) * 1000 if ref_fs > 0 else np.inf,
        # Fraction of video frames whose sync index actually lands inside the reference.
        'ref_coverage': float(((idx >= 0) & (idx < len(v))).mean()),
        'idx_monotonic': bool(np.all(np.diff(idx) >= 0)),
        'ecg_fixed_point': bool(np.all(ecg % ECG_SCALE == 0)),
        'ecg_amp': float((ecg.max() - ecg.min()) / ECG_SCALE),
        'dev_hr_med': float(np.nanmedian(dev)) if np.isfinite(dev).any() else np.nan,
        'dev_hr_valid': float(np.isfinite(dev).mean()),
        'ppg_live': bool(len(np.unique(v['PPG'].to_numpy())) > 1),
        'spo2_live': bool(len(np.unique(v['SpO2'].to_numpy())) > 1),
    }
    # A session is excluded only when its reference cannot be read at all.
    faults = []
    if row['dropped_rows']:
        faults.append(f"{row['dropped_rows']} unparsable row(s)")
    if not (REF_FS_RANGE[0] <= ref_fs <= REF_FS_RANGE[1]):
        faults.append(f'reference rate {ref_fs:.3g} Hz')
    if not row['ecg_fixed_point']:
        faults.append('ECG not fixed point')
    row['fault'] = '; '.join(faults)
    row['usable'] = not faults
    return row


timing = pd.DataFrame([audit_session(r.subject, r.activity)
                       for r in sessions.itertuples() if r.sync and r.viatom])
excluded = timing.loc[~timing.usable, ['subject', 'activity', 'fault']]
timing = timing[timing.usable].reset_index(drop=True)
GOOD_SESSIONS = list(zip(timing.subject, timing.activity))

n_frames_total = int(timing.n_frames.sum())
print(f'{len(timing) + len(excluded)} sessions read from the csv files alone, no video decoded')
print('excluded, reference unreadable:',
      excluded.to_dict('records') if len(excluded) else 'none')
print(f'{len(timing)} sessions carried forward')
print()
print(f'frames per video  : {timing.n_frames.min()}-{timing.n_frames.max()} over '
      f'{timing.span_s.min():.2f}-{timing.span_s.max():.2f} s')
print(f'measured fps      : {timing.fps.min():.3f}-{timing.fps.max():.3f} '
      f'(median {timing.fps.median():.3f}) | below the {MIN_FPS_ACCEPT:g} fps gate: '
      f'{int((timing.fps < MIN_FPS_ACCEPT).sum())} | below {MIN_FPS_WARN:g}: '
      f'{int((timing.fps < MIN_FPS_WARN).sum())}')
print(f'frame interval    : median {timing.dt_med_ms.median():.1f} ms, worst '
      f'{timing.dt_max_ms.max():.0f} ms | intervals over 50 ms: {int(timing.dt_gt_50ms.sum())} '
      f'of {n_frames_total} ({timing.dt_gt_50ms.sum() / n_frames_total:.2%})')
print(f'uniform-grid error: rms {timing.grid_rms_ms.median():.1f} ms median, '
      f'{timing.grid_rms_ms.max():.1f} ms worst session | peak deviation '
      f'{timing.grid_max_ms.max():.0f} ms = '
      f'{timing.grid_max_ms.max() * timing.fps.median() / 1000:.1f} frames')
print()
print(f'reference rate    : {timing.ref_fs.min():.2f}-{timing.ref_fs.max():.2f} Hz '
      f'(nominal {VIATOM_FS_NOMINAL:g})')
print(f'reference span    : {timing.ref_span_s.min():.1f}-{timing.ref_span_s.max():.1f} s '
      f'around a {timing.span_s.median():.0f} s video')
print(f'video starts      : {timing.offset_s.min():.1f}-{timing.offset_s.max():.1f} s into '
      f'the reference (median {timing.offset_s.median():.1f}) - never assume zero')
print(f'sync vs clocks    : worst disagreement {timing.clock_gap_ms.max():.0f} ms | '
      f'monotonic in {int(timing.idx_monotonic.sum())}/{len(timing)}')
print(f'reference covers  : {timing.ref_coverage.min():.3f}-{timing.ref_coverage.max():.3f} of '
      f'frames (sessions below 1.000: {int((timing.ref_coverage < 1).sum())}) - frames whose '
      f'sync index falls past the end of the reference have no label')
print(f'ECG amplitude     : {timing.ecg_amp.min():.0f}-{timing.ecg_amp.max():.0f} scaled units')
print()
print(f'live PPG columns  : {int(timing.ppg_live.sum())} sessions | '
      f'live SpO2: {int(timing.spo2_live.sum())}')
print(f'device HR         : per-session median {timing.dev_hr_med.min():.0f}-'
      f'{timing.dev_hr_med.max():.0f} bpm after unwrapping | readings inside the '
      f'physiological band: {timing.dev_hr_valid.min():.2f}-{timing.dev_hr_valid.max():.2f}')

101 sessions read from the csv files alone, no video decoded
excluded, reference unreadable: [{'subject': '07', 'activity': '02', 'fault': '1 unparsable row(s)'}]
100 sessions carried forward

frames per video  : 1800-1800 over 59.94-63.89 s
measured fps      : 28.158-30.016 (median 30.000) | below the 27 fps gate: 0 | below 20: 0
frame interval    : median 32.0 ms, worst 960 ms | intervals over 50 ms: 5815 of 180000 (3.23%)
uniform-grid error: rms 10.8 ms median, 2226.4 ms worst session | peak deviation 3860 ms = 115.8 frames

reference rate    : 124.97-125.09 Hz (nominal 125)
reference span    : 68.8-1427.2 s around a 60 s video
video starts      : 8.8-1367.3 s into the reference (median 18.7) - never assume zero
sync vs clocks    : worst disagreement 36 ms | monotonic in 100/100
reference covers  : 0.987-1.000 of frames (sessions below 1.000: 21) - frames whose sync index falls past the end of the reference have no label
ECG amplitude     : 130-3468 scaled units

live PPG columns  :

## 4. Reference heart rate: beats, not spectra
Two rules from earlier corpora apply directly here.

**A whole-recording spectral peak is not a heart-rate reference.** On VitalVideos it put
a clean contact PPG more than 3 bpm from the simultaneous ECG in 14 of 300 recordings,
purely because a heart rate that drifts across the minute spreads its peak; beat
intervals cut that to 2. On a corpus where subjects are exercising, the drift is the
whole point - so the reference is built from R-peaks, and the readout is the median beat
interval, which an occasional missed beat barely moves.

**Windows are keyed by their start frame.** Estimators drop windows they cannot read, so
pairing by position in a list compares a window against the wrong reference - the bug
fixed across the app on 2026-09-15. Every row below carries `window_start`, on the app's
own grid of `CLIP_LEN` frames at `WINDOW_STRIDE`.

**One detector is not enough on this corpus, and the first attempt proved it.** A plain
Pan-Tompkins threshold, which is adequate on resting data, loses beats once the rate
climbs and the QRS amplitude starts moving: on 09/02 it returned 82 bpm where the device
said 153, and the autocorrelation of the same QRS energy said 153.1. A reference that
fails on the fast recordings is useless here, because the fast recordings are the reason
this corpus is worth running.

So the reference is a **consensus of three readouts** that fail in different ways:

- **beat intervals** from the thresholded QRS energy - sharp when it works, and it
  carries beat timing nothing else does;
- **the autocorrelation of the QRS energy** - no threshold to miscalibrate, so it holds
  up at high rate, but it can lock onto a half or double period;
- **the device's own `ECG HR`**, unwrapped - independent hardware, but smoothed and
  therefore late when the rate is changing quickly.

A window gets a reference heart rate only when **at least two of the three agree within
3 bpm**, the same bar NB_P3_27 used between VitalVideos' PPG, ECG and CMS readouts, and
the value is the median of those that agree. Across the corpus this accepts 90% of
windows, with 83 of 100 recordings above 80% accepted and none at zero - so the gate
separates recordings rather than eliminating them. The two ECG readouts agree with each
other to 0.66 bpm and with the device to 2.7-3.8 bpm, which is the signature of a
smoothed device readout lagging a changing rate rather than of a bad detector.

The agreement counts are themselves the finding to carry into the protocol: a recording
where the three readouts cannot be made to agree does not have a trustworthy reference,
and a refusal there says nothing about the app. Nothing is gated on a model number yet -
the protocol has to be written before the first one is seen.

In [5]:
def qrs_energy(ecg, fs):
    """Band-passes the ECG to the QRS band, squares it and integrates over 120 ms.

    Squaring makes the result independent of lead polarity, which the dataset does not
    document and which is not constant across subjects.
    """
    x = np.asarray(ecg, dtype=np.float64)
    b, a = butter(2, [QRS_LOW_HZ / (fs / 2), QRS_HIGH_HZ / (fs / 2)], btype='band')
    win = max(1, int(0.12 * fs))
    return np.convolve(filtfilt(b, a, x - x.mean()) ** 2, np.ones(win) / win, mode='same')


def rate_from_peaks(energy, fs):
    """Heart rate from the median interval between thresholded QRS-energy peaks.

    Sharp where it works, and the only one of the three readouts that carries beat
    timing. It loses beats when the rate is high and the amplitude is moving, which is
    why it is one vote rather than the answer.
    """
    if energy.size < int(2 * fs):
        return np.nan, np.nan
    threshold = np.median(energy) + 0.5 * (np.percentile(energy, 98) - np.median(energy))
    peaks, _ = find_peaks(energy, height=threshold,
                          distance=max(1, int(60.0 / HR_MAX_BPM * fs)))
    ibi = np.diff(peaks) / fs
    ibi = ibi[(ibi > 60.0 / HR_MAX_BPM) & (ibi < 60.0 / HR_MIN_BPM)]
    if ibi.size < 3:
        return np.nan, np.nan
    med = float(np.median(ibi))
    return 60.0 / med, float(np.median(np.abs(ibi - med)) / med)


def rate_from_autocorr(energy, fs):
    """Heart rate from the dominant period of the QRS energy, with its peak height.

    The autocorrelation is divided by the overlap count at each lag, because the raw
    sum shrinks with lag and would bias every window towards a short period, which is a
    high heart rate. The peak height is a periodicity confidence, not a quality score.
    """
    x = np.asarray(energy, dtype=np.float64) - np.mean(energy)
    if x.size < 32 or x.std() < 1e-12:
        return np.nan, 0.0
    ac = np.correlate(x, x, 'full')[x.size - 1:] / np.arange(x.size, 0, -1)
    if ac[0] <= 0:
        return np.nan, 0.0
    ac = ac / ac[0]
    lo, hi = int(60.0 / HR_MAX_BPM * fs), min(int(60.0 / HR_MIN_BPM * fs), len(ac) - 2)
    if hi <= lo + 1:
        return np.nan, 0.0
    k = lo + int(np.argmax(ac[lo:hi + 1]))
    height = float(ac[k])
    if 0 < k < len(ac) - 1:
        curve = ac[k - 1] - 2 * ac[k] + ac[k + 1]
        if abs(curve) > 1e-12:
            k = k + 0.5 * (ac[k - 1] - ac[k + 1]) / curve
    return float(60.0 * fs / k), height


def consensus_hr(values, tol=HR_AGREE_BPM):
    """Returns (HR, how many readouts agreed) for the largest cluster within `tol`.

    Two agreeing readouts are required. Three readouts that fail in different ways
    rarely agree on the same wrong answer, which is what makes this stronger than any
    one of them, and the count is kept so a one-vote window can be excluded rather than
    quietly trusted.
    """
    finite = [v for v in values if np.isfinite(v)]
    best = []
    for i in range(len(finite)):
        group = [v for v in finite if abs(v - finite[i]) <= tol]
        if len(group) > len(best):
            best = group
    return (float(np.median(best)), len(best)) if len(best) >= 2 else (np.nan, len(best))


def reference_windows(subject, activity):
    """Per-window reference HR on the app's window grid, keyed by start frame."""
    folder = ECGF_ROOT / subject / activity
    t_frame, idx = read_sync(folder / 'c920.csv')
    v = read_viatom(folder / 'viatom-raw.csv')
    t_ref = v['milliseconds'].to_numpy()
    fs = (len(v) - 1) / ((t_ref[-1] - t_ref[0]) / 1000)
    energy = qrs_energy(v['ECG'].to_numpy() / ECG_SCALE, fs)
    dev = device_hr(v)
    rows = []
    for k in range(0, len(t_frame) - CLIP_LEN + 1, WINDOW_STRIDE):
        lo = int(np.clip(idx[k], 0, len(v) - 1))
        hi = int(np.clip(idx[k + CLIP_LEN - 1], 0, len(v) - 1))
        seg = energy[lo:hi + 1]
        hr_peaks, disp = rate_from_peaks(seg, fs)
        hr_auto, periodicity = rate_from_autocorr(seg, fs)
        window_dev = dev[lo:hi + 1]
        hr_device = (float(np.nanmedian(window_dev))
                     if np.isfinite(window_dev).any() else np.nan)
        hr, agree = consensus_hr([hr_peaks, hr_auto, hr_device])
        rows.append({'subject': subject, 'activity': activity, 'window_start': k,
                     'hr_bpm': hr, 'agree': agree, 'hr_peaks': hr_peaks,
                     'hr_auto': hr_auto, 'hr_device': hr_device, 'rr_disp': disp,
                     'periodicity': periodicity,
                     'covered': bool(0 <= idx[k] and idx[k + CLIP_LEN - 1] < len(v))})
    return rows


ref = pd.DataFrame([row for s, a in GOOD_SESSIONS for row in reference_windows(s, a)])
rec = (ref.groupby(['subject', 'activity'])
          .agg(windows=('hr_bpm', 'size'),
               accepted=('agree', lambda s: int((s >= 2).sum())),
               hr_med=('hr_bpm', 'median'),
               hr_spread=('hr_bpm', lambda s: s.quantile(0.75) - s.quantile(0.25)),
               disp_med=('rr_disp', 'median'),
               periodicity=('periodicity', 'median'))
          .reset_index())
rec['accepted_frac'] = rec.accepted / rec.windows

print(f'{len(ref)} windows of {CLIP_LEN / TARGET_FPS:.2f} s over {len(rec)} recordings')
print(f'windows with a reference (two readouts within {HR_AGREE_BPM:g} bpm): '
      f'{int((ref.agree >= 2).sum())} ({(ref.agree >= 2).mean():.1%}) | '
      f'all three agreeing: {int((ref.agree == 3).sum())} ({(ref.agree == 3).mean():.1%})')
print(f'windows whose sync index runs past the end of the reference: '
      f'{int((~ref.covered).sum())}')
print()
print('pairwise agreement, median absolute difference over windows where both are finite:')
for x, y in (('hr_peaks', 'hr_auto'), ('hr_peaks', 'hr_device'), ('hr_auto', 'hr_device')):
    both = ref[[x, y]].dropna()
    print(f'  {x:9s} vs {y:9s}: {(both[x] - both[y]).abs().median():5.2f} bpm '
          f'over {len(both)} windows')
print()
print(f'recordings with 80% or more of windows accepted: '
      f'{int((rec.accepted_frac >= 0.8).sum())}/{len(rec)} | 50% or more: '
      f'{int((rec.accepted_frac >= 0.5).sum())} | none accepted: '
      f'{int((rec.accepted_frac == 0).sum())}')
print('\nweakest references, worst 8 (these are where a refusal would mean nothing):')
print(rec.nsmallest(8, 'accepted_frac')[['subject', 'activity', 'windows', 'accepted',
                                         'accepted_frac', 'hr_med', 'periodicity']]
      .round(3).to_string(index=False))
print()
print('reference HR by activity (recording medians):')
print(rec.groupby('activity').hr_med.describe()[['count', 'min', '50%', 'max']].round(1).to_string())
print()
accepted = ref[ref.agree >= 2]
print(f'windows above 120 bpm: {int((accepted.hr_bpm > 120).sum())} '
      f'({(accepted.hr_bpm > 120).mean():.1%}) | above 140: '
      f'{int((accepted.hr_bpm > 140).sum())}')
print(f'recordings whose median HR is above 120 bpm: {int((rec.hr_med > 120).sum())} of {len(rec)}')
print('for context, the rest of the Phase-3 corpus holds 23 recordings above 120 bpm in total,')
print('none of them with sustained motion (MCD 42-132, UBFC-rPPG 54-133, PhysDrive 54-138,')
print('UBFC-Phys 46-106, VitalVideos 48-120).')

2100 windows of 5.33 s over 100 recordings
windows with a reference (two readouts within 3 bpm): 1883 (89.7%) | all three agreeing: 928 (44.2%)
windows whose sync index runs past the end of the reference: 0

pairwise agreement, median absolute difference over windows where both are finite:
  hr_peaks  vs hr_auto  :  0.66 bpm over 2078 windows
  hr_peaks  vs hr_device:  2.69 bpm over 2078 windows
  hr_auto   vs hr_device:  3.77 bpm over 2100 windows

recordings with 80% or more of windows accepted: 83/100 | 50% or more: 98 | none accepted: 0

weakest references, worst 8 (these are where a refusal would mean nothing):
subject activity  windows  accepted  accepted_frac  hr_med  periodicity
     08       02       21         5          0.238 136.057        0.272
     09       02       21         5          0.238 153.570        0.769
     08       04       21        11          0.524 137.476        0.937
     00       02       21        13          0.619 152.015        0.870
     09       04

## 5. Face geometry and motion, from the authors' boxes
The `bbox` files give a per-frame face box for almost every video, which answers three
questions at the cost of reading a few megabytes.

**Which camera.** Phase 2 took `c920-1` as the frontal, tripod-mounted one and `c920-2`
as machine-mounted, without recording a comparison. Median face size and the fraction of
frames with a detection settle it.

**Whether the crop contract survives.** The app's box is square in *relative*
coordinates, so on a 16:9 frame it is deliberately rectangular in pixels - training
applied the same stretch, and the aspect that matters is therefore the frame's 1.78,
which sits inside `TRAIN_ASPECT_RANGE`. What can still fail is clamping: a face near an
edge produces a padded box that falls outside the frame, and the app refuses a capture
whose box is clamped by more than `MAX_BOX_SIDE_LOST` of its side. The estimate here uses
the authors' detector box rather than MediaPipe landmark extremes, so it is a proxy;
section 7 checks the real thing on a subset.

**How much the head moves**, as the median displacement of the box centre between
frames. This is the only motion measure available without decoding, and it is what
distinguishes a seated talking session from a rowing machine - which is how the activity
ids can be characterised without trusting the Phase-2 names.

In [6]:
def face_audit(subject, activity, camera):
    """Face size, detection coverage, head motion and a crop-clamp proxy for one video."""
    path = BBOX_ROOT / subject / activity / f'{camera}.face'
    if not path.exists():
        return None
    a = np.loadtxt(path, dtype=np.float64)
    w, h = a[:, 3], a[:, 4]
    found = (w > 0) & (h > 0)
    if found.sum() < 2:
        return {'subject': subject, 'activity': activity, 'camera': camera,
                'rows': len(a), 'found_frac': float(found.mean())}
    cx, cy = a[:, 1] + w / 2, a[:, 2] + h / 2
    step = np.hypot(np.diff(cx[found]), np.diff(cy[found]))
    # The contract box: square in relative coordinates, expanded by CROP_PADDING.
    rel_w, rel_h = np.median(w[found]) / FRAME_W, np.median(h[found]) / FRAME_H
    half = max(rel_w, rel_h) * (1.0 + CROP_PADDING) / 2.0
    rcx, rcy = np.median(cx[found]) / FRAME_W, np.median(cy[found]) / FRAME_H
    box = (rcx - half, rcy - half, rcx + half, rcy + half)
    over_px = ((max(0.0, -box[0]) + max(0.0, box[2] - 1.0)) * FRAME_W
               + (max(0.0, -box[1]) + max(0.0, box[3] - 1.0)) * FRAME_H)
    return {'subject': subject, 'activity': activity, 'camera': camera,
            'rows': len(a), 'found_frac': float(found.mean()),
            'face_w_px': float(np.median(w[found])), 'face_h_px': float(np.median(h[found])),
            'box_cx_rel': float(np.median(cx[found])) / FRAME_W,
            'box_cy_rel': float(np.median(cy[found])) / FRAME_H,
            'box_square_px': bool(np.allclose(w[found], h[found])),
            'crop_w_px': 2 * half * FRAME_W, 'crop_h_px': 2 * half * FRAME_H,
            'face_fill': float(np.median(h[found]) / (2 * half * FRAME_H)),
            'frac_side_lost': float(over_px / (2 * half * FRAME_W)),
            'motion_px_med': float(np.median(step)), 'motion_px_p95': float(np.percentile(step, 95))}


faces = pd.DataFrame([r for s in sessions.itertuples() for c in CAMERAS
                      if (r := face_audit(s.subject, s.activity, c)) is not None])

print(f'{len(faces)} videos with author boxes | frames per file: '
      f'{faces.rows.min()}-{faces.rows.max()}')
print(f'frames with a detection: {faces.found_frac.min():.3f}-{faces.found_frac.max():.3f} '
      f'(median {faces.found_frac.median():.3f})')
if faces.found_frac.min() == 1.0:
    print('  the authors\' detector found a face in every frame of every video, so this '
          'column separates nothing;')
    print('  the detector question is settled in section 7, on our own contract.')
print()
print('by camera (medians):')
print(faces.groupby('camera')[['face_w_px', 'face_h_px', 'box_cx_rel', 'box_cy_rel',
                               'found_frac', 'motion_px_med', 'motion_px_p95',
                               'frac_side_lost']]
      .median().round(3).to_string())
wider = faces.groupby('camera').face_w_px.median().idxmax()
calmer = faces.groupby('camera').motion_px_med.median().idxmin()
print(f'  larger face: {wider} | less apparent head motion: {calmer}')
print()
# The author box is about as wide as it is tall in pixels, and the frame is 16:9, so the
# relative height always exceeds the relative width: the padded box is driven by height in
# every video and face-to-crop height is pinned at 1/(1+CROP_PADDING), carrying no
# information. What varies is the face size in real pixels, which is what the 72x72 resize
# has to work from. (The per-frame boxes are not square - widths run 242-353 px against a
# near-equal median height - so a squareness test is the wrong diagnostic here.)
height_drives = (faces.face_h_px / FRAME_H >= faces.face_w_px / FRAME_W)
print(f'padded box driven by face height in {int(height_drives.sum())}/{len(faces)} videos | '
      f'face / crop height therefore pinned at {faces.face_fill.median():.3f}, '
      f'about {TARGET_SIZE * faces.face_fill.median():.0f} of {TARGET_SIZE} px after the resize')
print(f'crop the contract would take from a 1920x1080 frame: '
      f'{faces.crop_w_px.median():.0f}x{faces.crop_h_px.median():.0f} px '
      f'(aspect {faces.crop_w_px.median() / faces.crop_h_px.median():.2f}, '
      f'training range {TRAIN_ASPECT_RANGE[0]}-{TRAIN_ASPECT_RANGE[1]})')
print(f'videos whose padded box is clamped by more than {MAX_BOX_SIDE_LOST:.0%} of its side: '
      f'{int((faces.frac_side_lost > MAX_BOX_SIDE_LOST).sum())} of {len(faces)}')
print()
print('head motion and face size by activity (c920-1, medians):')
print(faces[faces.camera == 'c920-1']
      .groupby('activity')[['motion_px_med', 'motion_px_p95', 'face_w_px']]
      .median().round(2).to_string())
print()
print('smallest faces, worst 6 (least real detail before the 72x72 resize):')
print(faces.nsmallest(6, 'face_w_px')[['subject', 'activity', 'camera', 'face_w_px',
                                       'motion_px_med', 'frac_side_lost']]
      .round(2).to_string(index=False))
print('\nmost head motion, worst 6:')
print(faces.nlargest(6, 'motion_px_med')[['subject', 'activity', 'camera', 'motion_px_med',
                                          'motion_px_p95', 'face_w_px']]
      .round(2).to_string(index=False))
print(f'\nvideos clamped by more than {MAX_BOX_SIDE_LOST:.0%}:')
clamped = faces.loc[faces.frac_side_lost > MAX_BOX_SIDE_LOST,
                    ['subject', 'activity', 'camera', 'frac_side_lost', 'face_w_px']]
print(clamped.round(3).to_string(index=False) if len(clamped) else '  none')

200 videos with author boxes | frames per file: 1800-1800
frames with a detection: 1.000-1.000 (median 1.000)
  the authors' detector found a face in every frame of every video, so this column separates nothing;
  the detector question is settled in section 7, on our own contract.

by camera (medians):
        face_w_px  face_h_px  box_cx_rel  box_cy_rel  found_frac  motion_px_med  motion_px_p95  frac_side_lost
camera                                                                                                        
c920-1      259.5      259.5       0.528       0.389         1.0          6.954         18.302             0.0
c920-2      294.0      294.0       0.507       0.408         1.0          3.536         13.543             0.0
  larger face: c920-2 | less apparent head motion: c920-2

padded box driven by face height in 200/200 videos | face / crop height therefore pinned at 0.625, about 45 of 72 px after the resize
crop the contract would take from a 1920x1080 frame: 772x43

## 6. The halogen exclusion, tested
NB_P2_01 dropped activities 03 and 04 from the corpus with this reason: *"Activities 03
and 04 use halogen lighting which causes 50 Hz flicker. At 30 fps this aliases to 3.33 Hz
- inside our rPPG bandpass (0.7-3.5 Hz)."* A third of the dataset was discarded on that
sentence, and it was never measured.

The arithmetic does not hold. Aliasing folds a frequency about multiples of the sampling
rate: a component at `f` sampled at `fps` appears at `|f - round(f / fps) * fps|`. At
30 fps, 50 Hz folds to `|50 - 2*30| = 10 Hz` and the 100 Hz brightness ripple an
incandescent lamp actually produces - brightness peaks twice per mains cycle - folds to
`|100 - 3*30| = 10 Hz`. Both land near 10 Hz, well above the 3.0 Hz top of the cardiac
band, where the band-pass removes them. 3.33 Hz is 50/15, which is the Nyquist frequency,
not an alias. Thermal inertia in a halogen filament damps the ripple further, and an
exposure of 10 ms or longer integrates most of what is left.

Two things could still make halogen recordings bad in a way that matters, and neither is
aliasing: auto-exposure hunting under a flickering source, which produces slow brightness
swings that *do* sit in the cardiac band, and a beat frequency if the true frame rate is
not exactly 30 fps. Section 3 measured the real rate, so the predicted alias can be
computed per recording rather than assumed.

This matters in the opposite direction too. The published sample frames show the ambient
sessions are **dark** - Phase 2's own pipeline skipped recordings on a brightness floor -
while the halogen ones are bright and warm. If the better-lit half of the corpus was
dropped for an arithmetic error, a refusal benchmark built on the remainder is being run
on the hardest lighting the dataset has, and would report a refusal rate that says as
much about the exclusion as about the app.

**This cell decodes video.** It reads `PROBE_FRAMES` frames from each activity of one
subject: at 1920x1080 uncompressed that is about 3.1 MB per frame, so the default of 600
frames is roughly 1.9 GB per video and a few minutes for the six over USB. Start with one
subject, read the output, and widen only if the answer is ambiguous. The background patch
is a wall region chosen for subject 15's framing - check it before reusing it on another
subject, and treat a non-skin control as evidence about illumination only, never as proof
that a region carries no pulse (the crop-corner control failed for exactly that reason).

In [7]:
PROBE_SUBJECTS = ('00', '08', '15')   # three subjects; one is not evidence about a corpus
PROBE_CAMERA = 'c920-1'
PROBE_START, PROBE_FRAMES = 300, 600  # 20 s starting 10 s in; ~1.9 GB read per video
CARDIAC_LOW_HZ, CARDIAC_HIGH_HZ = 0.66, 3.0 # the app's readout band
ALIAS_LOW_HZ, ALIAS_HIGH_HZ = 8.0, 12.0 # where mains flicker is predicted to land
PATCH_W, PATCH_Y = 0.15, (0.10, 0.60) # background patch: width and vertical extent


def background_patch(boxes):
    """Picks a wall patch on whichever side of the frame is furthest from the face.

    Hand-picking a rectangle per subject does not survive contact with a second subject,
    and a patch that catches the shoulder or a machine measures the subject, not the
    light. The author boxes give the face position for free, so the side is chosen from
    the data and reported with its spatial texture: a patch that is genuinely blank wall
    has a low pixel standard deviation, and a high one means the choice should be checked.
    """
    found = (boxes[:, 3] > 0) & (boxes[:, 4] > 0)
    face_cx = float(np.median(boxes[found, 1] + boxes[found, 3] / 2)) / FRAME_W
    left = (0.02, 0.02 + PATCH_W)
    right = (0.98 - PATCH_W, 0.98)
    side = left if face_cx > 0.5 else right
    name = 'left' if face_cx > 0.5 else 'right'
    return (int(side[0] * FRAME_W), int(side[1] * FRAME_W),
            int(PATCH_Y[0] * FRAME_H), int(PATCH_Y[1] * FRAME_H), name, face_cx)


def luminance_traces(video, boxes, patch, start=PROBE_START, n=PROBE_FRAMES):
    """Mean luminance of the face box and of the wall patch, frame by frame.

    No frame is rendered or kept; three scalars per frame are all that survives the loop.
    """
    bx0, bx1, by0, by1, _, _ = patch
    cap = cv2.VideoCapture(str(video))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    face, back, texture = [], [], []
    for i in range(start, start + n):
        ok, frame = cap.read()
        if not ok:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        patch_px = gray[by0:by1, bx0:bx1]
        back.append(float(patch_px.mean()))
        texture.append(float(patch_px.std()))
        if i < len(boxes) and boxes[i, 3] > 0 and boxes[i, 4] > 0:
            x, y, w, h = boxes[i, 1:5].astype(int)
            face.append(float(gray[max(0, y):y + h, max(0, x):x + w].mean()))
        else:
            face.append(np.nan)
    cap.release()
    return np.asarray(face), np.asarray(back), float(np.mean(texture)) if texture else np.nan


def band_fraction(x, fps, low, high):
    """Fraction of a trace's a.c. power falling between two frequencies.

    Gaps where the face box is missing are interpolated rather than dropped: dropping
    them shortens the trace and shifts every frequency on the axis, which would be a
    quiet way to invent or hide a spectral line. A trace missing more than a tenth of
    its samples is refused instead.
    """
    x = np.asarray(x, dtype=np.float64)
    ok = np.isfinite(x)
    if x.size < 32 or ok.mean() < 0.9:
        return np.nan
    if not ok.all():
        x = np.interp(np.arange(x.size), np.flatnonzero(ok), x[ok])
    p = np.abs(np.fft.rfft((x - x.mean()) * np.hanning(x.size))) ** 2
    f = np.fft.rfftfreq(x.size, 1.0 / fps)
    total = p[f > 0].sum()
    return float(p[(f >= low) & (f <= high)].sum() / total) if total > 0 else np.nan


rows = []
for subject in PROBE_SUBJECTS:
    for act in sorted(timing.loc[timing.subject == subject, 'activity']):
        folder = ECGF_ROOT / subject / act
        video = folder / f'{PROBE_CAMERA}.avi'
        bbox_path = BBOX_ROOT / subject / act / f'{PROBE_CAMERA}.face'
        if not (video.exists() and bbox_path.exists()):
            continue
        fps = float(timing.loc[(timing.subject == subject) & (timing.activity == act), 'fps'].iloc[0])
        boxes = np.loadtxt(bbox_path, dtype=np.float64)
        patch = background_patch(boxes)
        face, back, texture = luminance_traces(video, boxes, patch)
        rows.append({
            'subject': subject, 'activity': act, 'fps': fps,
            'alias_hz': abs(2 * MAINS_HZ - round(2 * MAINS_HZ / fps) * fps),
            'patch': patch[4], 'patch_sd': texture,
            'face_lum': float(np.nanmean(face)), 'back_lum': float(np.mean(back)),
            'back_cardiac': band_fraction(back, fps, CARDIAC_LOW_HZ, CARDIAC_HIGH_HZ),
            'back_alias': band_fraction(back, fps, ALIAS_LOW_HZ, ALIAS_HIGH_HZ),
            'face_cardiac': band_fraction(face, fps, CARDIAC_LOW_HZ, CARDIAC_HIGH_HZ),
            'face_alias': band_fraction(face, fps, ALIAS_LOW_HZ, ALIAS_HIGH_HZ)})

probe = pd.DataFrame(rows)
BRIGHT_ACTIVITIES = ('03', '04')   # the pair Phase 2 excluded as halogen
probe['lighting'] = np.where(probe.activity.isin(BRIGHT_ACTIVITIES), 'halogen', 'ambient')

print(f'subjects {list(PROBE_SUBJECTS)} | {PROBE_CAMERA} | {PROBE_FRAMES} frames from frame {PROBE_START}')
print(f'mains {MAINS_HZ:g} Hz -> lamp ripple {2 * MAINS_HZ:g} Hz -> predicted alias '
      f'{probe.alias_hz.min():.2f}-{probe.alias_hz.max():.2f} Hz, against a cardiac band of '
      f'{CARDIAC_LOW_HZ}-{CARDIAC_HIGH_HZ} Hz')
print(f'background patch chosen away from the face: {probe.patch.value_counts().to_dict()} | '
      f'pixel sd {probe.patch_sd.min():.1f}-{probe.patch_sd.max():.1f} '
      f'(low means blank wall; a high value means check that recording)')
print()
print(probe[['subject', 'activity', 'lighting', 'fps', 'alias_hz', 'patch', 'patch_sd',
             'back_lum', 'face_lum', 'back_cardiac', 'back_alias', 'face_cardiac']]
      .round(4).to_string(index=False))

subjects ['00', '08', '15'] | c920-1 | 600 frames from frame 300
mains 50 Hz -> lamp ripple 100 Hz -> predicted alias 9.95-12.63 Hz, against a cardiac band of 0.66-3.0 Hz
background patch chosen away from the face: {'left': 12, 'right': 6} | pixel sd 2.8-15.0 (low means blank wall; a high value means check that recording)

subject activity lighting     fps  alias_hz patch  patch_sd  back_lum  face_lum  back_cardiac  back_alias  face_cardiac
     00       01  ambient 30.0093    9.9720  left    4.9715   55.0802   26.5142        0.0311      0.0004        0.0208
     00       02  ambient 30.0068    9.9795  left    5.6366   58.0795   20.3672        0.7185      0.0006        0.8328
     00       03  halogen 29.9893   10.0320  left    5.9485   98.1192   43.8237        0.0209      0.0092        0.1358
     00       04  halogen 30.0018    9.9945  left    6.3507   95.9276   36.0478        0.8507      0.0021        0.5569
     00       05  ambient 29.9878   10.0365 right    7.4728   41.5059   53.

In [8]:
# The matched comparison. Activities pair as 01/03 and 02/04 - the same activity under two
# lightings, confirmed by head motion in section 5 (2.1 against 2.6 px, 7.7 against 7.4) -
# so each pair holds the activity fixed and varies only the lamp. Grouping 03/04 against
# everything else instead confounds lighting with motion, and that confound is what made
# subject 00 look like a counterexample: its wall in-band power is 0.72 on ambient 02 and
# 0.85 on halogen 04, which is the activity, not the light.
LIGHTING_PAIRS = [('01', '03'), ('02', '04')]   # (ambient, halogen), same activity

pairs = []
for subject, g in probe.groupby('subject'):
    by_act = g.set_index('activity')
    for amb, hal in LIGHTING_PAIRS:
        if amb not in by_act.index or hal not in by_act.index:
            continue
        a, h = by_act.loc[amb], by_act.loc[hal]
        pairs.append({'subject': subject, 'pair': f'{amb}->{hal}',
                      'lum_ratio': h.back_lum / a.back_lum,
                      'cardiac_amb': a.back_cardiac, 'cardiac_hal': h.back_cardiac,
                      'alias_amb': a.back_alias, 'alias_hal': h.back_alias})
pairs = pd.DataFrame(pairs)
pairs['d_cardiac'] = pairs.cardiac_hal - pairs.cardiac_amb
pairs['d_alias'] = pairs.alias_hal - pairs.alias_amb

print('matched pairs: same activity, ambient -> halogen')
print(pairs.round(4).to_string(index=False))
print()
print(f'brighter under halogen in {int((pairs.lum_ratio > 1).sum())}/{len(pairs)} pairs, '
      f'x{pairs.lum_ratio.min():.2f}-x{pairs.lum_ratio.max():.2f}')
print(f'wall cardiac-band power falls in {int((pairs.d_cardiac < 0).sum())}/{len(pairs)} pairs '
      f'(median {pairs.d_cardiac.median():+.4f})')
print(f'largest alias-band fraction anywhere in the probe: {probe.back_alias.max():.4f} '
      f'({probe.loc[probe.back_alias.idxmax(), "subject"]}/'
      f'{probe.loc[probe.back_alias.idxmax(), "activity"]}), against the +0.21 to +0.29 a '
      f'synthetic 9.98 Hz ripple produced as a positive control')
print()
print('what the wall patch actually tracks - in-band power by activity, median over subjects:')
print(probe.groupby('activity').back_cardiac.median().round(4).to_string())
print('  it follows head motion (section 5), not the lamp: the two calmest activities sit')
print('  lowest whichever light they were shot under.')

matched pairs: same activity, ambient -> halogen
subject   pair  lum_ratio  cardiac_amb  cardiac_hal  alias_amb  alias_hal  d_cardiac  d_alias
     00 01->03     1.7814       0.0311       0.0209     0.0004     0.0092    -0.0101   0.0088
     00 02->04     1.6517       0.7185       0.8507     0.0006     0.0021     0.1322   0.0015
     08 01->03     1.2940       0.0190       0.0003     0.0027     0.0000    -0.0187  -0.0027
     08 02->04     1.2591       0.4023       0.0052     0.0002     0.0000    -0.3971  -0.0002
     15 01->03     2.3362       0.0113       0.0395     0.0001     0.0000     0.0282  -0.0000
     15 02->04     2.4668       0.1843       0.0715     0.0003     0.0004    -0.1128   0.0001

brighter under halogen in 6/6 pairs, x1.26-x2.47
wall cardiac-band power falls in 4/6 pairs (median -0.0144)
largest alias-band fraction anywhere in the probe: 0.0092 (00/03), against the +0.21 to +0.29 a synthetic 9.98 Hz ripple produced as a positive control

what the wall patch actually t

## 7. Does our own detector find these faces?
Phase 2 recorded that subjects 01, 03, 06, 11 and 14 passed nothing, with the note that
face detection failed for them throughout. That verdict came from a different pipeline -
a 468-point face mesh with anatomical ROIs, under a brightness floor - so it does not
transfer to the current contract, and it matters a great deal which it was. A recording
the detector cannot see contributes a refusal that measures MediaPipe; a recording it
sees but the estimator cannot read contributes a refusal that measures the estimator.
Only the second belongs in the benchmark headline.

The check imports the app's own `face_box` module rather than reimplementing it, as
NB_P3_27 did for its crop, and samples `N_DETECT_FRAMES` frames per recording - the same
number the contract uses to build its median stable box. Seeking in an uncompressed AVI
is cheap, so this reads about 37 MB per recording.

Section 5 already reported whether the *authors'* detector found a face in those
recordings. Where the two disagree, the data is fine and our detector is the limit.

In [9]:
from app.live_vitals.preprocess.face_box import (box_clamp_report, landmark_box,
                                                 make_landmarker, square_padded_box)

# Phase 2's failures, plus two controls: a subject it had no trouble with, and one
# halogen session to see whether lighting rather than the subject was the problem.
PARITY_SESSIONS = [('01', '01'), ('03', '01'), ('06', '01'), ('11', '01'), ('14', '01'),
                   ('00', '01'), ('15', '01'), ('15', '03')]
LANDMARKER = make_landmarker()


def detector_parity(subject, activity, camera='c920-1', n=N_DETECT_FRAMES):
    """Runs the app's own detector on evenly sampled frames and reports the crop implied."""
    folder = ECGF_ROOT / subject / activity
    video = folder / f'{camera}.avi'
    if not video.exists():
        return {'subject': subject, 'activity': activity, 'detected': np.nan, 'of': n}
    total = len(read_sync(folder / 'c920.csv')[0])
    cap = cv2.VideoCapture(str(video))
    boxes = []
    for i in np.linspace(0, total - 1, n).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame = cap.read()
        if not ok:
            continue
        box = landmark_box(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), LANDMARKER)
        if box is not None:
            boxes.append(box)
    cap.release()
    row = {'subject': subject, 'activity': activity, 'detected': len(boxes), 'of': n}
    if not boxes:
        return row
    padded = square_padded_box(boxes)
    report = box_clamp_report(padded, FRAME_W, FRAME_H)
    landmark_h = float(np.median([b[3] - b[1] for b in boxes]))
    row.update({'face_fill': landmark_h / (padded[3] - padded[1]),
                'frac_side_lost': report['frac_side_lost'],
                'expected_aspect': report['expected_aspect'],
                'aspect_ok': report['aspect_in_training_range'],
                'clamped': report['clamped']})
    return row


parity = pd.DataFrame([detector_parity(s, a) for s, a in PARITY_SESSIONS])
parity = parity.merge(faces.loc[faces.camera == 'c920-1', ['subject', 'activity', 'found_frac']],
                      on=['subject', 'activity'], how='left')
parity = parity.rename(columns={'found_frac': 'author_found_frac'})

print(f'app detector on {N_DETECT_FRAMES} sampled frames per recording, '
      f'beside the authors\' own detection rate')
print(parity.round(3).to_string(index=False))
print()
print(f'crop-clamp gate: MAX_BOX_SIDE_LOST = {MAX_BOX_SIDE_LOST:.0%} | '
      f'aspect accepted between {TRAIN_ASPECT_RANGE[0]} and {TRAIN_ASPECT_RANGE[1]}')

app detector on 12 sampled frames per recording, beside the authors' own detection rate
subject activity  detected  of  face_fill  frac_side_lost  expected_aspect  aspect_ok  clamped  author_found_frac
     01       01        11  12      0.631             0.0            1.778       True    False                1.0
     03       01        12  12      0.600             0.0            1.778       True    False                1.0
     06       01        10  12      0.612             0.0            1.778       True    False                1.0
     11       01        10  12      0.625             0.0            1.778       True    False                1.0
     14       01        10  12      0.637             0.0            1.778       True    False                1.0
     00       01        12  12      0.609             0.0            1.778       True    False                1.0
     15       01        12  12      0.623             0.0            1.778       True    False                1.0


## 8. Findings and decision

**Layout and formats (confirmed).** 17 subjects x 6 activities x 2 cameras: 101 session
folders, 202 videos, 1127 GB of uncompressed 1920x1080 YUV 4:2:0, 1800 frames each.
Session 02/04 has no video. Session 07/02 has no author bounding box and a truncated
reference file, so the authors excluded it and so does this notebook: **100 usable
sessions**. Each session carries two clocks - `c920.csv` is two columns, a wall-clock
timestamp per frame and a row index into `viatom-raw.csv`, and the reference runs
68.8-1427.2 s around a 60 s video, starting 8.8-1367.3 s before it (median 18.7).
Alignment goes through the sync column; proportional resampling, correct for UBFC-rPPG,
would shift every label here by however long the operator waited. The ECG column is fixed
point, every value a multiple of 2^16, and the reference rate measures 124.97-125.09 Hz.

**There is no PPG anywhere.** `PPG`, `PPG HR`, `SpO2` and `PI` are dead constants in all
100 usable sessions; the one apparently live PPG in the first run was the truncated row in
07/02. The evaluation-only verdict now rests on measurement: PhysNet v2's loss wants a
waveform and this corpus has none, so ECG-Fitness cannot enter training under the current
architecture.

**Frame timing.** Measured from the capture clock, not the container: 28.158-30.016 fps,
median 30.000, every session above the 27 fps gate. Inside that nominal 30 fps, 3.23% of
frame intervals exceed 50 ms and the worst reaches 960 ms. Deviation from the uniform grid
a Phase-3 store would impose is 10.8 ms rms in the median session but 2226 ms in the worst,
peaking at 3860 ms - 116 frames. Storing frames uniformly is safe for a spectral heart-rate
readout in most recordings and not in all: carry `grid_rms_ms` per recording into the store,
and make no beat-timing claim from this corpus.

**The reference, and the gate it suggests.** A single thresholded R-peak detector is not
adequate here - on 09/02 it read 82 bpm where both the device and the autocorrelation of the
same QRS energy said 153. The reference is a consensus of three readouts that fail
differently, accepted when two agree within 3 bpm: **1883 of 2100 windows (89.7%)**, with 83
of 100 recordings above 80% accepted and none at zero. The two ECG readouts agree with each
other to 0.66 bpm; the device sits 2.69 and 3.77 bpm away, the signature of a smoothed
readout lagging a changing rate rather than of a bad detector. The device still earns its
vote: 15.3% of accepted windows are accepted only because it agrees with one ECG readout,
and it independently confirms 42.4% of those where the two ECG readouts already agree.

Candidate gate for the protocol: a window needs two readouts within 3 bpm, a recording needs
80% of its windows. That admits 83 recordings. The weakest are 08/02 and 09/02, both at 0.24,
and they fail differently - 08/02 has a periodicity of 0.27, so there is little periodic
structure to find, while 09/02 is strongly periodic at 0.77 and the readouts simply disagree.
Both belong in the write-up as recordings where a refusal by the app would say nothing about
the app.

**Heart-rate coverage.** 531 windows (28.2%) above 120 bpm and 179 above 140; 31 of 100
recordings carry a median above 120, up to 163. The entire rest of the Phase-3 corpus holds
23 recordings above 120 bpm, none with sustained motion. The caveat survives the measurement:
high rate and violent motion arrive together here, so this extends coverage without isolating
a high-HR bias. MCD-rPPG's post-exercise recordings, subject still, remain the cleaner
instrument for that question.

**Camera: c920-2.** It carries a larger face (294 against 259 px median) and half the
frame-to-frame box motion (3.54 against 6.95 px). Those pull in opposite directions for a
closer camera, which magnifies any head movement, so the comparison is made per unit face
size: 0.0120 against 0.0268 px of motion per px of face, **c920-2 steadier by 2.2x**. The
published sample frames for subject 15 agree - c920-2 is a level, near-frontal
head-and-shoulders shot while c920-1 sits further back and higher. The two cameras frame the
subject almost identically otherwise (box centre 0.507, 0.408 against 0.528, 0.389), so this
is distance and steadiness, not a different mounting. **Phase 2 had it backwards** when it
took c920-1 as the tripod-mounted frontal camera and discarded c920-2 as non-frontal.

**Crop geometry.** The padded box is driven by face height in every video - the author box is
near-square in pixels on a 16:9 frame - so face-to-crop height is pinned at 0.625, about 45 of
72 px after the resize, and carries no information. What varies is absolute face size:
169 px at the smallest (12/01) against 614 px at the largest. The crop the contract takes is
772x434 px at aspect 1.78, inside the training range. 11 of 200 videos clamp beyond 2%, worst
0.103, and **every one of them is on activity 05 or 06** - the two highest-motion activities,
where the face is both largest and closest to a frame edge.

**Activity structure, measured here and confirmed by the authors.** Head motion splits the six
ids into three pairs - 01/03 at 2.1 and 2.6 px per frame, 02/04 at 7.7 and 7.4, 05/06 at 11.2
and 12.9 - with reference HR medians of 83/92, 112/125 and 115/111, and across three subjects 03
and 04 are 1.26x to 2.47x brighter than their partners. The authors' description of the dataset,
as quoted in a 2025 Scientific Reports study that used it (doi:10.1038/s41598-025-06031-8),
matches that structure from the other side: 17 subjects performing four activities - speaking,
rowing, a stationary bike and an elliptical trainer - with speaking and rowing each recorded
twice, "once in the presence of halogen light ... and the other in its absence", under three
lighting setups in all: window light, a 400 W halogen lamp and a 30 W LED. So 01/03 and 02/04
are speaking and rowing, and 05 and 06 are the bike and the elliptical. The data point to which
pair is which: 01/03 carries the least head motion and the lowest heart rate of the six, which
fits speaking, making 02/04 rowing - the reverse of NB_P2_01's labels, which had 01 as rowing and
02 as speaking. Nothing measured here says which of 05 and 06 is the bike. "Ambient" in the
store means "not halogen", and covers both the window light and the LED.

**The halogen exclusion is refuted; activities 03 and 04 belong in the benchmark.** The authors
describe the halogen as producing "a strong 50 Hz (100 Hz) temporal interference", and that is
true of the light. What Phase 2 got wrong is where it lands in a 30 fps video: the 100 Hz ripple
aliases to 9.95-12.63 Hz at the measured rates, far above the 3.0 Hz top of the readout band,
and 3.33 Hz is 50/15, the Nyquist frequency, not an alias. Measured on matched activity pairs
across three subjects, halogen is brighter in **6 of 6** pairs, and the wall patch shows
**almost none of the interference**: its largest 8-12 Hz fraction in eighteen recordings is
0.0092, against +0.21 to +0.29 from a synthetic 9.98 Hz ripple injected as a positive control,
so the measurement is sensitive to the thing it looked for. Cardiac-band power on the wall falls
in 4 of 6 pairs (median -0.0144). Phase 2 discarded a third of the corpus, and the better-lit
third, on an arithmetic error about where a real interference would appear.

**What the wall patch found instead.** In-band power on a non-skin patch reaches 0.85 and 0.72
on subject 00 and 0.57 on subject 15, always on the high-motion activities, and by activity it
tracks head motion rather than the lamp. **Motion, not illumination, is what puts energy into
the band the readout searches.** That is a prediction the benchmark can test: the windows where
the app reports confidently and wrongly should concentrate on activities 05 and 06.

**Our detector sees these faces.** The Phase-2 verdict that subjects 01, 03, 06, 11 and 14 have
no findable face is withdrawn. The authors' detector found a face in every frame of all 200
videos, and MediaPipe under the current contract found one in 10 to 12 of 12 sampled frames on
every recording tested, those five included, with nothing clamped and the aspect in range. A
refusal in this corpus would be the estimator's, not the detector's - which is what makes the
benchmark meaningful.

**Decision.** Build a Phase-3 store from **c920-2**, all 100 usable sessions including 03 and
04, 1800 frames of 72x72 uint8 per recording - about 2.8 GB - with `has_bvp=False`,
`has_ecg=True`, the consensus reference heart rate and its agreement count per window, and
`grid_rms_ms` per recording. Then write the refusal-benchmark protocol **before any model
number is seen**: primary metric the fraction of windows and recordings the app declines,
reported per activity and never pooled; the failure that matters is a window the app calls
`ok` and misses by more than 10 bpm; recordings below the reference gate reported separately.

**Three defects**, all of the house type - nothing raised an
error and each produced a plausible wrong number. `c920.csv` is two columns, and reading it
flattened interleaves a reference index with a 13-digit epoch timestamp. A truncated final line
in 07/02 gives that file a 1.4e10 s span and makes its dead PPG column look alive. The Viatom
`ECG HR` column is a signed byte that wraps above 127, so the independent cross-check reads
negative on 45 recordings, all of them the fast ones.

**Still open.** Which of 05 and 06 is the bike and which the elliptical, and a confirmation of
the speaking (01/03) and rowing (02/04) reading above, both from the folder table in the
dataset's own documentation. The dataset page (http://cmp.felk.cvut.cz/~spetlrad/ecg-fitness/)
serves plain HTTP only and could not be read when this was written. The clamped videos are no
longer open: NB_P3_29 writes them with their `app_frac_side_lost` and `capture_verdict`, and its
protocol reports them separately.